# Module 16 — MLOps for Protein ML

## TL;DR — Plain English

**What is MLOps?** MLOps bridges the gap between "my notebook runs" and "this runs reliably in production for a team." It is the set of practices that make ML systems reproducible, auditable, and deployable at scale.

### The Gap: Notebook vs Production

| Notebook | Production |
|----------|------------|
| Run once on your laptop | Runs 1000x/day on servers |
| You remember what you did | Others need to reproduce your results |
| Any result is fine | Wrong predictions cost money or patient health |
| Dependencies informal | Pinned versions, containers |
| No monitoring | Alerts when model degrades |

### Five Pillars of MLOps

1. **Reproducibility** — Same code + same data + same seed → same results. Requires: config files, seed management, data checksums, pinned dependencies.
2. **Experiment Tracking** — Which hyperparameters gave the best AUC? Which protein family was hardest? Tools: MLflow, Weights & Biases, TensorBoard.
3. **Model Serving** — An API endpoint that takes a protein sequence and returns a prediction. Tools: FastAPI + uvicorn, TorchServe, HuggingFace Inference Endpoints.
4. **Monitoring** — Did my model's accuracy drop? Are the input sequences from a different distribution than training? Tools: Prometheus + Grafana, custom KS-test monitors.
5. **CI/CD** — Every code change is automatically tested and deployed. Tools: GitHub Actions, Jenkins, DVC.

### Why Protein ML Specifically Needs MLOps

- **Expensive GPU runs**: AlphaFold3 training costs ~\$1M. You cannot redo experiments carelessly — every run must be logged.
- **Multiple model versions**: AF2, AF3, ESM2-8M, ESM2-15B, ESMFold, OpenFold — tracking which version predicted which structure is critical for wet lab follow-up.
- **Wet lab validation cycles**: A computational prediction today is validated experimentally in 6–12 weeks. You need an audit trail: which model, which weights, which input → which prediction → which experiment.
- **Regulatory pressure**: Clinical applications (drug discovery) require model cards, uncertainty estimates, and reproducibility documentation.
- **Distribution shift**: Training on UniProt; production sees novel protein families, pathogen variants, synthetic proteins — monitoring is essential.

### What You Will Be Able to Do After This Notebook

- Track any experiment with MLflow (parameters, metrics, artifacts)
- Write a fully reproducible training script (seeds, configs, checksums)
- Checkpoint models with metadata and auto-keep-top-k
- Build a FastAPI REST endpoint for model serving
- Write a Dockerfile and docker-compose for GPU-enabled ML services
- Detect data drift with the Kolmogorov-Smirnov test
- Design a GitHub Actions CI/CD pipeline for an ML model
- Answer senior-level MLOps interview questions

## Beginner Teaching Frame

**Who should fully work through this notebook:** students who already know how to train models and now need to think like an ML systems engineer.

**How to study it on a first pass:**
- focus on reproducibility, checkpointing, serving, and monitoring
- treat specific tools as examples of deeper systems ideas
- ask what can break after training is over

**Common traps:** thinking MLOps is just Docker, or treating deployment as separate from model quality and data quality.


## Canonical Resource Upgrade

Strong references here:
- [Made With ML](https://madewithml.com/)
- [Full Stack Deep Learning](https://fullstackdeeplearning.com/)
- [MLflow docs](https://mlflow.org/docs/latest/index.html)
- [FastAPI tutorial](https://fastapi.tiangolo.com/tutorial/)


## Cross-References

### Prerequisites
- **00/06 PyTorch Fundamentals** — model classes, state_dict, optimizers (used in checkpointing section)
- **00/02 ML Fundamentals** — metrics (AUC, accuracy), train/val/test splits, pipelines (assumed throughout)

### Directly Related Modules
- **10/01 Protein Structure Fine-Tuning** — the LoRA-fine-tuned ESM2 model built there is exactly what you would deploy using this notebook's patterns
- **07/01 AlphaFold3 Core** — production AlphaFold3 inference is the canonical example of large model serving; the monitoring patterns here apply to AF3 predictions
- **11/01 Membrane Protein Dynamics** — LoRA adapters trained per protein family can be served using the multi-adapter pattern described here
- **15/01 Self-Supervised Learning** — ESM2 pre-training pipelines require MLflow tracking at scale; reproducibility is critical

### Position in Curriculum

```
Modules 00-15: Learn the science and build the models
       |
       v
Module 16 (this notebook): How to ship everything you built
```

This is the **final notebook** in the curriculum. It teaches the engineering practices that turn research models into production systems — the skill that separates senior ML engineers from researchers at companies like Isomorphic Labs and DeepMind.

In [ ]:
import os
import hashlib
import json
import random
import numpy as np
import torch

# Reproducibility: seeds, configs, checksums
def set_all_seeds(seed=42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"All seeds set to {seed}")

set_all_seeds(42)

# Config management
config = {
    "model": {"d_model": 256, "n_heads": 8, "n_layers": 4, "dropout": 0.1},
    "training": {"lr": 1e-4, "batch_size": 32, "epochs": 100, "seed": 42},
    "data": {"dataset": "skempi_v2", "train_split": 0.8, "normalize": True},
    "lora": {"rank": 8, "alpha": 16, "dropout": 0.05},
}
# Compute config hash for experiment tracking
config_hash = hashlib.md5(json.dumps(config, sort_keys=True).encode()).hexdigest()[:8]
print(f"\nConfig hash: {config_hash}")
print(f"Config: {json.dumps(config, indent=2)[:200]}...")

# Data checksum
def compute_file_checksum(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()[:16]

# Simulate checksum check
print(f"\nReproducibility checklist:")
print(f"  ✓ Seed: {config['training']['seed']}")
print(f"  ✓ Config hash: {config_hash}")
print(f"  ✓ CUDA deterministic: {torch.backends.cudnn.deterministic}")
print(f"  ✓ torch version: {torch.__version__}")

In [ ]:
import mlflow
import torch
import torch.nn as nn
import numpy as np

# MLflow experiment tracking
print("MLflow Experiment Tracking for Protein ML")
print("=" * 50)

# Set up MLflow (in-memory for demo)
mlflow.set_tracking_uri("file:///tmp/mlruns")
mlflow.set_experiment("protein_ddg_prediction")

class SimpleModel(nn.Module):
    def __init__(self, input_dim=20, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x)

torch.manual_seed(42)
model = SimpleModel()
X = torch.randn(100, 20); y = torch.randn(100, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Log experiment
with mlflow.start_run(run_name="baseline_ridge_v1") as run:
    # Log parameters
    mlflow.log_params({
        "model_type": "SimpleModel",
        "hidden_dim": 64,
        "lr": 1e-3,
        "seed": 42,
    })

    # Training loop with logging
    for epoch in range(5):
        pred = model(X)
        loss = nn.MSELoss()(pred, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        pearson_r = np.corrcoef(pred.detach().numpy().flatten(), y.numpy().flatten())[0,1]
        mlflow.log_metrics({"train_loss": loss.item(), "pearson_r": pearson_r}, step=epoch)

    # Log model artifact
    mlflow.pytorch.log_model(model, "model")
    print(f"Run ID: {run.info.run_id[:8]}...")
    print(f"Final loss: {loss.item():.4f}")
    print(f"Final Pearson r: {pearson_r:.4f}")
    print(f"Logged to: /tmp/mlruns")

In [ ]:
import torch
import torch.nn as nn
import os

# Checkpointing — save/load training state
def save_checkpoint(model, optimizer, epoch, loss, path, metadata=None):
    """Save full training checkpoint (model + optimizer state)."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'metadata': metadata or {},
    }
    torch.save(checkpoint, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"Checkpoint saved: epoch={epoch}, loss={loss:.4f}, size={size_mb:.2f}MB")

def load_checkpoint(path, model, optimizer=None):
    """Resume training from checkpoint."""
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    print(f"Loaded checkpoint: epoch={ckpt['epoch']}, loss={ckpt['loss']:.4f}")
    return ckpt['epoch'], ckpt['loss']

# Demo
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train for 3 steps
X = torch.randn(32, 20); y = torch.randn(32, 1)
for step in range(3):
    loss = nn.MSELoss()(model(X), y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

save_checkpoint(model, optimizer, epoch=3, loss=loss.item(),
                path='/tmp/model_epoch3.pt',
                metadata={'dataset': 'skempi_v2', 'metric': 'mse'})

# Simulate resuming
model2 = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
epoch_resumed, loss_resumed = load_checkpoint('/tmp/model_epoch3.pt', model2, opt2)
print(f"Resumed from epoch {epoch_resumed}")

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import torch
import torch.nn as nn
import numpy as np
import uvicorn
import threading

# FastAPI serving for protein ΔΔG prediction
app = FastAPI(title="Protein ΔΔG Predictor API")

class MutationRequest(BaseModel):
    sequence: str
    position: int
    wildtype: str
    mutant: str

class DDGResponse(BaseModel):
    ddg: float
    confidence: float
    interpretation: str

# Load model (simulated)
torch.manual_seed(42)
predictor = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))

@app.get("/health")
def health():
    return {"status": "ok", "model": "ddg_predictor_v1"}

@app.post("/predict", response_model=DDGResponse)
def predict(request: MutationRequest):
    # Simulate feature extraction
    features = torch.zeros(1, 20)
    features[0, hash(request.wildtype) % 20] = 1.0
    features[0, hash(request.mutant) % 20] -= 0.5

    with torch.no_grad():
        ddg = predictor(features).item()

    interpretation = "Stabilizing" if ddg < -0.5 else "Destabilizing" if ddg > 0.5 else "Neutral"
    return DDGResponse(ddg=round(ddg, 3), confidence=0.75, interpretation=interpretation)

# Demo: test the API endpoint logic directly
req = MutationRequest(sequence="ACDEFGHIKLM", position=5, wildtype="A", mutant="V")
result = predict(req)
print(f"API Response:")
print(f"  ΔΔG: {result.ddg} kcal/mol")
print(f"  Confidence: {result.confidence}")
print(f"  Interpretation: {result.interpretation}")
print()
print("Deployment: uvicorn main:app --host 0.0.0.0 --port 8000")
print("Docker: FROM python:3.10 + pip install fastapi uvicorn torch + COPY + CMD")

In [ ]:
import numpy as np
from scipy import stats

# Data drift monitoring for protein ML models
print("Production Drift Monitoring for Protein ML")
print("=" * 60)
print()

class DriftMonitor:
    """Monitor input feature distribution drift using KS test."""
    def __init__(self, reference_data, feature_names=None, threshold=0.05):
        self.reference = reference_data
        self.feature_names = feature_names or [f'feat_{i}' for i in range(reference_data.shape[1])]
        self.threshold = threshold

    def check_drift(self, new_data):
        n_features = self.reference.shape[1]
        drift_report = []
        for i in range(n_features):
            ks_stat, p_val = stats.ks_2samp(self.reference[:, i], new_data[:, i])
            drifted = p_val < self.threshold
            drift_report.append({
                'feature': self.feature_names[i],
                'ks_stat': round(ks_stat, 4),
                'p_value': round(p_val, 4),
                'drifted': drifted
            })
        return drift_report

np.random.seed(42)
# Training distribution (SKEMPI mutations)
ref_data = np.random.randn(1000, 10)
# Production data: slight drift (new mutations from different proteins)
prod_data = np.random.randn(200, 10) + np.array([0.5,0,0,0.3,0,0,0,0.2,0,0])

monitor = DriftMonitor(ref_data, threshold=0.05)
report = monitor.check_drift(prod_data)
n_drifted = sum(r['drifted'] for r in report)
print(f"Features with drift: {n_drifted}/{len(report)}")
print()
for r in report:
    flag = "⚠ DRIFT" if r['drifted'] else "OK"
    print(f"  {r['feature']}: KS={r['ks_stat']:.4f} p={r['p_value']:.4f} {flag}")
print()
print("Action when drift detected:")
print("  1. Alert on-call ML engineer")
print("  2. Log incoming data for retraining")
print("  3. Consider rolling back to previous model version")

In [ ]:
import subprocess
import json

# CI/CD pipeline for ML models
print("CI/CD Pipeline for Protein ML")
print("=" * 60)
print()

pipeline_config = {
    "name": "protein-ml-ci",
    "trigger": ["push to main", "pull request"],
    "stages": [
        {
            "name": "1. Code Quality",
            "steps": ["flake8 src/", "black --check src/", "mypy src/"]
        },
        {
            "name": "2. Unit Tests",
            "steps": ["pytest tests/ -v --tb=short", "pytest --cov=src --cov-report=xml"]
        },
        {
            "name": "3. Model Validation",
            "steps": [
                "python validate_model.py --dataset val_set.csv --metric pearson_r",
                "assert pearson_r >= 0.65  # performance threshold"
            ]
        },
        {
            "name": "4. Integration Tests",
            "steps": [
                "docker build -t protein-ml:test .",
                "docker run protein-ml:test python -m pytest tests/integration/",
            ]
        },
        {
            "name": "5. Deploy (on merge to main)",
            "steps": [
                "docker push registry/protein-ml:latest",
                "kubectl set image deployment/ddg-api ddg-api=protein-ml:${SHA}",
                "kubectl rollout status deployment/ddg-api"
            ]
        },
    ]
}

for stage in pipeline_config["stages"]:
    print(f"Stage: {stage['name']}")
    for step in stage["steps"]:
        print(f"  $ {step}")
    print()

print("Best practices:")
practices = [
    "Never push model weights to git (use DVC or MLflow)",
    "Pin dependency versions (requirements.txt + pip-compile)",
    "Test with small model variant in CI (fast, no GPU needed)",
    "Gate deployment on validation metric threshold",
    "Blue-green deployment: route 10% traffic to new model first",
]
for p in practices:
    print(f"  • {p}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# Gradient accumulation for large-batch training on limited GPU
print("Gradient Accumulation: Simulate Large Batches")
print("=" * 60)

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Effective batch size = micro_batch * accumulation_steps
micro_batch = 8
accum_steps = 4
effective_batch = micro_batch * accum_steps
print(f"Micro batch: {micro_batch}, Accumulation: {accum_steps}×")
print(f"Effective batch size: {effective_batch} (simulates GPU with {effective_batch}× more memory)")

losses = []
for step in range(3):  # 3 "effective" gradient steps
    optimizer.zero_grad()
    total_loss = 0
    for micro_step in range(accum_steps):
        X = torch.randn(micro_batch, 256)
        y = torch.randn(micro_batch, 1)
        pred = model(X)
        loss = nn.MSELoss()(pred, y) / accum_steps  # normalize by accum steps
        loss.backward()  # accumulate gradients
        total_loss += loss.item()
    # Clip gradients and update
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    losses.append(total_loss)
    print(f"  Step {step+1}: loss={total_loss:.4f}")

print(f"\nGradient accumulation enables training with effective batch={effective_batch}")
print("Critical for Pairformer fine-tuning where single protein barely fits on GPU")

## Interview Q&A

---

**Q: What is train-serve skew and why is it dangerous?**

A: Train-serve skew occurs when features are computed differently at training time versus serving time. Example: training uses `np.mean(all_sequences)` over the full dataset; serving computes mean over a single sample — giving a different normalisation. The model sees out-of-distribution inputs even for normal sequences, causing silent accuracy degradation (no error, just wrong predictions). Solution: a **feature store** (Feast, Tecton, AWS SageMaker Feature Store) that centralises feature computation so both training and serving call identical code.

---

**Q: How would you detect if your protein stability model has degraded in production?**

A: Use a layered monitoring strategy:
1. **Input drift**: KS test on amino acid frequencies, sequence length histogram, protein family distribution (cluster with ESM2 embeddings).
2. **Output drift**: histogram of predicted ΔΔG scores shifts toward extreme values.
3. **Shadow testing**: run both old and new model; flag cases where predictions disagree by >1 kcal/mol.
4. **Slice-based evaluation**: track AUC separately per protein family (soluble vs membrane vs IDR).
5. **Labelled sentinel set**: 50 proteins with known experimental ΔΔG values; evaluate weekly.

---

**Q: MLflow vs Weights & Biases vs TensorBoard — when to use each?**

| Tool | Best For | Limitations |
|------|----------|-------------|
| TensorBoard | Quick training visualisation; built into PyTorch/TF | Metrics only; no artifact management |
| MLflow | Full experiment lifecycle (params, metrics, artifacts, model registry); self-hostable | UI less polished than W&B |
| W&B | Best UX; team collaboration; protein structure visualisation | Paid at scale; data leaves your servers |

For bioinformatics at Isomorphic Labs / DeepMind: **MLflow preferred** — self-hosted, protein sequence data stays on-prem, integrates with SLURM-based GPU clusters.

---

**Q: How do you serve a 15B parameter model (ESM2-15B) at production latency?**

A:
1. **Quantisation**: INT8 via `bitsandbytes` or `torch.quantization` → 4× memory reduction, ~2× speedup.
2. **vLLM / TGI**: paged KV-cache attention enables continuous batching → higher throughput.
3. **Model parallelism**: tensor parallelism across multiple A100s using `deepspeed` or `megatron`.
4. **LoRA adapter serving**: keep base model once in memory, swap per-task LoRA adapters at request time (adds ~1 ms).
5. **Distillation**: train ESM2-8M (43M params) to mimic ESM2-15B via knowledge distillation → 350× smaller.

---

**Q: What is a feature store and when does protein ML need one?**

A: A feature store is a centralised system that computes, stores, and serves ML features consistently. Both the training pipeline and serving pipeline call the same feature store API — eliminating train-serve skew. Protein ML needs feature stores when:
- MSA computation is expensive (hours per sequence) and should be cached
- Multiple models share the same ESM2 embeddings
- Feature definitions change: you need to version features separately from model weights

Tools: Feast (open-source), Tecton (managed), AWS SageMaker Feature Store.

---

**Q: What is a model card and what must it include for a protein ML model?**
A: A model card (Mitchell et al., 2019) documents: (1) intended use and out-of-scope uses, (2) training data composition and exclusions, (3) performance metrics across relevant subgroups (e.g., membrane vs soluble proteins), (4) known failure modes, (5) ethical considerations. For protein ML: always specify which protein families were excluded from training — a model trained only on soluble proteins will silently fail on membrane proteins.

---

**Q: What is the difference between aleatoric uncertainty and OOD detection?**
A: Aleatoric: inherent noise in the data (measurement error). Can be captured by a model that outputs a distribution. OOD: the input is outside the training distribution — the model shouldn't be trusted at all. OOD detection uses features like Mahalanobis distance (embedding space), maximum softmax probability, or energy score to flag inputs the model has never seen.

---

**Q: What is Expected Calibration Error (ECE) and how do you fix a miscalibrated model?**
A: ECE = mean(|accuracy_in_bin - confidence_in_bin|) over confidence bins. A perfectly calibrated model has ECE=0. Fix: temperature scaling — divide logits by T before softmax. T > 1 softens predictions (reduces overconfidence). T is found by minimizing NLL on a calibration set. One of the cheapest and most effective post-hoc fixes.

---

## Key MLOps Tools Cheat Sheet

| Need | Tool | Bioinformatics Note |
|------|------|---------------------|
| Experiment tracking | MLflow, W&B | W&B has native protein structure viz |
| Data versioning | DVC, LakeFS | Track MSA files (multi-GB) |
| Model registry | MLflow, HuggingFace Hub | HF Hub for open protein models |
| Serving | FastAPI + uvicorn, TorchServe | FastAPI for prototyping; TorchServe for production |
| Containers | Docker, Singularity | Singularity for HPC (no root required) |
| Orchestration | Kubernetes, SLURM | SLURM for academic GPU clusters |
| Monitoring | Prometheus + Grafana | Monitor GPU utilisation, p99 latency |
| Feature store | Feast | Important for repeated MSA featurisation |

---
## Section 8 — Responsible AI: Model Cards, OOD Detection & Calibration

Production ML at Isomorphic Labs, DeepMind, and Google requires more than accuracy — models must know what they don't know, and their limitations must be documented.

**Model Cards (Mitchell et al., 2019):** Standardized documentation for ML models:
- What was the training data? What was excluded?
- What populations/proteins does the model work well on? Poorly?
- Known failure modes and edge cases
- Intended vs prohibited use cases

Isomorphic Labs publishes model cards for every deployed model. DeepMind's AlphaFold has a model card. This is non-negotiable for production deployment.

**Out-of-Distribution (OOD) Detection:** Your protein stability model was trained on soluble globular proteins. A membrane protein arrives → model confidently predicts wrong answer. OOD detection flags this.

**Calibration:** A well-calibrated model: when it says "90% confidence", it's right 90% of the time. Most neural networks are overconfident. Temperature scaling fixes this in one line.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

# Responsible AI: model cards, OOD detection, calibration
print("Responsible AI for Protein ML")
print("=" * 60)

# 1. Model Card
model_card = {
    "model_name": "ProteinDDG-LoRA-v1",
    "task": "ΔΔG prediction for single-point mutations",
    "input": "Protein sequence + mutation (wt_aa, position, mut_aa)",
    "output": "ΔΔG in kcal/mol (negative=stabilizing)",
    "training_data": "SKEMPI v2 (7,085 mutations across 319 proteins)",
    "limitations": [
        "Trained on soluble proteins; may not generalize to membrane proteins",
        "Poor extrapolation to mutations > 20 residues from training distribution",
        "Does not model epistatic effects (double mutations)",
    ],
    "performance": {"Pearson_r": 0.68, "RMSE": 1.2, "test_set": "SKEMPI held-out"},
}
print("\nModel Card:")
for k, v in model_card.items():
    if isinstance(v, list):
        print(f"  {k}:"); [print(f"    - {i}") for i in v]
    else:
        print(f"  {k}: {v}")

# 2. Calibration check
print("\nCalibration check (regression):")
torch.manual_seed(42)
y_true = np.random.randn(200)
y_pred = y_true + np.random.randn(200) * 0.5
pred_std = np.random.uniform(0.3, 0.8, 200)  # predicted uncertainty

# Reliability: 90% CI should contain true value 90% of time
ci_90 = np.abs(y_true - y_pred) < 1.645 * pred_std
print(f"  90% CI actual coverage: {ci_90.mean():.1%} (target: 90%)")
ci_50 = np.abs(y_true - y_pred) < 0.674 * pred_std
print(f"  50% CI actual coverage: {ci_50.mean():.1%} (target: 50%)")

# 3. OOD detection via Mahalanobis distance
print("\nOOD detection (Mahalanobis distance):")
ref_features = np.random.randn(500, 10)
mu = ref_features.mean(axis=0)
sigma_inv = np.linalg.inv(np.cov(ref_features.T) + 1e-6*np.eye(10))

in_dist_sample = np.random.randn(10)
out_dist_sample = np.random.randn(10) * 3 + 5  # OOD

def mahalanobis(x, mu, sigma_inv):
    diff = x - mu
    return np.sqrt(diff @ sigma_inv @ diff)

print(f"  In-distribution sample: score = {mahalanobis(in_dist_sample, mu, sigma_inv):.2f}")
print(f"  OOD sample: score = {mahalanobis(out_dist_sample, mu, sigma_inv):.2f}")
print("  Threshold: 99th percentile of calibration scores")

## Resources

### 1. Theory
- **"Designing Machine Learning Systems"** by Chip Huyen (O'Reilly, 2022) — the definitive MLOps book; covers data management, training pipelines, serving, monitoring
- **"Hidden Technical Debt in Machine Learning Systems"** (Sculley et al., Google, NeurIPS 2015): https://proceedings.neurips.cc/paper/2015/file/86df7dcfd896fcaf2674f757a2463eba-Paper.pdf
- **Continuous Delivery for Machine Learning** (Martin Fowler): https://martinfowler.com/articles/cd4ml.html
- **MLflow documentation**: https://mlflow.org/docs/latest/
- **"A Survey of Data Management for Machine Learning"** (VLDB 2021)

### 2. Must-Have Popular Resources

**Start Here (Zero Background):**
- **Made With ML** — free MLOps course by Goku Mohandas (best free curriculum): https://madewithml.com/
- **FastAPI tutorial** (official, 30 min to first API): https://fastapi.tiangolo.com/tutorial/
- **Docker for beginners** (Play With Docker labs): https://training.play-with-docker.com/
- **GitHub Actions quickstart**: https://docs.github.com/en/actions/quickstart

**Popular / Advanced:**
- **Full Stack Deep Learning** (MLOps-heavy, free): https://fullstackdeeplearning.com/
- **W&B courses** (free, with certificates): https://www.wandb.courses/
- **Chip Huyen ML Interviews Book** (systems section): https://huyenchip.com/ml-interviews-book/

### 3. Practicals
- Kaggle: "Deploy your first model with FastAPI + Docker" notebook template
- **HuggingFace Inference Endpoints** (1-click deployment): https://huggingface.co/inference-endpoints
- **W&B tutorial**: "Track your first experiment" (30 min): https://docs.wandb.ai/tutorials
- **AWS SageMaker free tier**: experiment tracking + deployment
- **DVC tutorial**: https://dvc.org/doc/start

### 4. Real-World Problems

1. **Protein structure prediction pipeline**: MLflow-tracked pipeline from sequence → ESM2 embeddings → structure classification → deployed FastAPI. Dataset: PDB sequences + SCOP labels.
2. **Drug discovery model monitoring**: Deploy ΔΔG prediction model; monitor for distribution shift when new protein targets are added. Dataset: ProteinGym substitutions.
3. **LoRA adapter serving**: Serve base ESM2 once in memory, swap per-protein-family LoRA adapters at request time. Benchmark latency vs one separate model per family.

### 5. Interview Tips
- Describe your full ML pipeline: data → features → train → eval → deploy → monitor
- For Isomorphic Labs / DeepMind: know HPC MLOps (SLURM, not just Kubernetes), large model serving (vLLM, TGI)
- Know the difference between **online** (real-time) and **batch** inference
- **"Train-serve skew"** is the most common MLOps interview topic — know it cold
- Mention reproducibility: seeds, config files, checksums — shows production mindset
- Know: model registry vs artifact store vs feature store (they are different things)

### 6. Hot Industry Topics
- **LLM serving infrastructure**: vLLM (paged attention, KV cache), TensorRT-LLM for GPU-optimised inference
- **LoRA adapter serving**: serve one base model + thousands of adapters (parameter-efficient multi-tenant ML)
- **Protein foundation model APIs**: ESMFold API (Meta), OpenFold inference server, Chai Discovery API
- **MLOps for biology**: wet-lab-in-the-loop pipelines, uncertainty-aware deployment (don't predict confidently on OOD proteins)
- **Model cards**: documenting protein model limitations, training data, evaluation populations (Isomorphic Labs publishes these for all production models)
- **FHE (Fully Homomorphic Encryption)**: run inference on encrypted patient sequences — emerging for clinical proteomics

## Timed Practice Problems

Work through these under exam conditions. Set a timer for each.

---

**Problem 1 — FastAPI endpoint validation (10 min)**

Write a Pydantic `PredictionRequest` model with these fields:
- `sequence: str` — validate it contains only standard amino acids (`ACDEFGHIKLMNPQRSTVWY`), converted to uppercase, max 1024 characters
- `wt_residue: str` — single character, must be a valid amino acid
- `mut_residue: str` — single character, must be a valid amino acid and different from `wt_residue`
- `position: int` — must be between 1 and `len(sequence)`

Write a `@validator` for each field. What error does FastAPI return if validation fails?

---

**Problem 2 — Drift detection (5 min)**

Describe three concrete ways to detect data drift in a production protein stability model. For each, state:
- What you measure
- What statistical test you use
- What action you take if drift is detected

---

**Problem 3 — MLflow experiment tracking vs model registry (3 min)**

What is the difference between:
- An **MLflow experiment** with logged runs
- The **MLflow Model Registry**

When would you promote a run from an experiment to the registry? What stages exist in the registry?

---

**Problem 4 — Docker deployment design (15 min — design question)**

Design a Docker-based deployment for a LoRA-fine-tuned ESM2 model with these requirements:
- Base ESM2-15B model is 30 GB; 5 per-family LoRA adapters are 50 MB each
- Must serve 3 protein families simultaneously
- P99 latency target: < 500 ms
- Zero-downtime deployments required

Design the Docker image(s), how adapters are loaded, and how you achieve zero-downtime updates.

---

**Problem 5 — EMA update for BYOL target network (3 min)**

BYOL (Bootstrap Your Own Latent) uses an exponential moving average (EMA) to update the target network. Implement this in 3 lines:

```python
# online_model and target_model are nn.Module instances
# tau = 0.996 (EMA decay)
# Your 3-line implementation here:
```

Hint: iterate over `zip(online_model.parameters(), target_model.parameters())`.

---

### Answers

<details>
<summary>Problem 5 answer (reveal after attempting)</summary>

```python
tau = 0.996
for online_p, target_p in zip(online_model.parameters(), target_model.parameters()):
    target_p.data = tau * target_p.data + (1 - tau) * online_p.data
```

The target network parameters are never updated by gradient descent — only by EMA. This prevents collapse in self-supervised learning.
</details>

## Mastery Check

Before moving on, make sure you can:
1. explain how to make an ML experiment reproducible
2. explain what train-serve skew is
3. explain why drift monitoring matters in biology-facing systems
4. describe a minimal safe deployment workflow


---
## 🚀 Production-Ready ML Serving

The basic FastAPI demo earlier works for single requests. Production systems need async serving, model versioning, and performance optimization.


In [ ]:
# PRODUCTION ML SERVING — Beyond the Demo

# 1. ASYNC FASTAPI (handles concurrent requests efficiently)
ASYNC_SERVER_CODE = '''
from fastapi import FastAPI, HTTPException, BackgroundTasks
from pydantic import BaseModel, validator
import asyncio
import torch
import numpy as np
import logging
import time
from typing import Optional, List
import hashlib

app = FastAPI(title="Protein ΔΔG Predictor", version="2.0")
logger = logging.getLogger(__name__)

# ── Model Registry ──
class ModelRegistry:
    """Simple model registry — in production use MLflow Model Registry."""
    def __init__(self):
        self.models = {}  # name -> {'model': ..., 'version': ..., 'metadata': ...}
        self.active = {}  # endpoint -> model_name

    def register(self, name, model, version, metadata=None):
        self.models[name] = {'model': model, 'version': version,
                              'metadata': metadata or {}, 'loaded_at': time.time()}
        logger.info(f"Registered model {name} v{version}")

    def set_active(self, endpoint, model_name):
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} not registered")
        self.active[endpoint] = model_name

    def get_active(self, endpoint):
        name = self.active.get(endpoint)
        if name is None or name not in self.models:
            raise RuntimeError(f"No active model for endpoint {endpoint}")
        return self.models[name]['model'], self.models[name]['version']

registry = ModelRegistry()

# ── Request/Response Models ──
class PredictRequest(BaseModel):
    sequence: str
    mutation: str  # e.g., "A42G" (Ala42Gly)

    @validator('sequence')
    def validate_sequence(cls, v):
        valid = set('ACDEFGHIKLMNPQRSTVWY')
        invalid = set(v.upper()) - valid
        if invalid:
            raise ValueError(f"Invalid amino acids: {invalid}")
        if len(v) > 500:
            raise ValueError("Sequence too long (max 500 aa)")
        return v.upper()

class PredictResponse(BaseModel):
    ddg: float
    uncertainty: Optional[float]
    model_version: str
    latency_ms: float
    request_id: str

# ── Prediction Endpoint (async) ──
@app.post("/predict/ddg", response_model=PredictResponse)
async def predict_ddg(request: PredictRequest):
    start = time.time()
    request_id = hashlib.md5(f"{request.sequence}{request.mutation}".encode()).hexdigest()[:8]

    try:
        model, version = registry.get_active("ddg")
    except RuntimeError as e:
        raise HTTPException(status_code=503, detail=str(e))

    # Run prediction in thread pool (non-blocking for async)
    import concurrent.futures
    loop = asyncio.get_event_loop()
    with concurrent.futures.ThreadPoolExecutor() as pool:
        ddg_pred = await loop.run_in_executor(pool, model, request.sequence)

    latency_ms = (time.time() - start) * 1000
    logger.info(f"Request {request_id}: {request.mutation} → ΔΔG={ddg_pred:.3f} ({latency_ms:.1f}ms)")

    return PredictResponse(
        ddg=float(ddg_pred),
        uncertainty=0.3,  # placeholder
        model_version=version,
        latency_ms=latency_ms,
        request_id=request_id,
    )

# ── Health + Readiness Probes (required by Kubernetes) ──
@app.get("/health")
async def health():
    return {"status": "ok", "timestamp": time.time()}

@app.get("/ready")
async def ready():
    if not registry.active:
        raise HTTPException(status_code=503, detail="No model loaded")
    return {"status": "ready", "active_models": registry.active}

# ── Model Hot-Swap (A/B testing) ──
@app.post("/registry/activate")
async def activate_model(endpoint: str, model_name: str):
    try:
        registry.set_active(endpoint, model_name)
        return {"status": "activated", "endpoint": endpoint, "model": model_name}
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))
'''

print("ASYNC FASTAPI SERVER CODE (production-ready):")
print("Key improvements over the demo:")
print("  1. Async endpoints: handle 100s of concurrent requests without blocking")
print("  2. Input validation with Pydantic validators")
print("  3. Model registry: register multiple versions, hot-swap without downtime")
print("  4. Health/readiness probes: required by Kubernetes orchestration")
print("  5. Request ID + structured logging: essential for debugging in production")
print()
print("Save to: protein_server.py")
print("Run with: uvicorn protein_server:app --host 0.0.0.0 --port 8080 --workers 4")

In [ ]:
# ONNX EXPORT — Fast CPU Inference (10-100x speedup over PyTorch)
# ONNX Runtime is the fastest way to serve PyTorch models in production

import torch
import torch.nn as nn
import numpy as np
import time

# Define a simple protein model
class ProteinPredictor(nn.Module):
    def __init__(self, vocab_size=21, embed_dim=64, seq_len=50):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Sequential(
            nn.Conv1d(embed_dim, 128, 5, padding=2), nn.ReLU(),
            nn.Conv1d(128, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        h = self.embed(x).transpose(1, 2)
        h = self.conv(h).squeeze(-1)
        return self.head(h).squeeze(-1)

model = ProteinPredictor()
model.eval()

# Dummy input
dummy_input = torch.randint(0, 20, (1, 50))  # (batch=1, seq_len=50)

# ── Export to ONNX ──
import os
onnx_path = '/tmp/protein_predictor.onnx'
torch.onnx.export(
    model, dummy_input, onnx_path,
    input_names=['sequence'],
    output_names=['ddg_prediction'],
    dynamic_axes={'sequence': {0: 'batch_size'}, 'ddg_prediction': {0: 'batch_size'}},
    opset_version=14,
)
print(f"ONNX model saved to {onnx_path}")

# ── Benchmark: PyTorch vs ONNX Runtime ──
n_batch = 100
batch_inputs = torch.randint(0, 20, (n_batch, 50))

# PyTorch inference
t0 = time.time()
with torch.no_grad():
    for i in range(n_batch):
        _ = model(batch_inputs[i:i+1])
torch_time = (time.time() - t0) * 1000 / n_batch

print(f"PyTorch inference: {torch_time:.3f} ms/sample")

try:
    import onnxruntime as ort

    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

    # Warm up
    for _ in range(10):
        sess.run(None, {'sequence': batch_inputs[:1].numpy()})

    # Benchmark ONNX
    t0 = time.time()
    for i in range(n_batch):
        _ = sess.run(None, {'sequence': batch_inputs[i:i+1].numpy()})
    onnx_time = (time.time() - t0) * 1000 / n_batch

    print(f"ONNX Runtime inference: {onnx_time:.3f} ms/sample")
    print(f"Speedup: {torch_time/onnx_time:.1f}×")

    # Verify outputs match
    pt_out = model(batch_inputs[:1]).item()
    onnx_out = sess.run(None, {'sequence': batch_inputs[:1].numpy()})[0][0]
    print(f"Output match: PyTorch={pt_out:.6f}, ONNX={onnx_out:.6f} "
          f"(diff={abs(pt_out-onnx_out):.2e})")

except ImportError:
    print("onnxruntime not installed. Install with: pip install onnxruntime")
    print()
    print("TYPICAL ONNX SPEEDUPS (CPU inference):")
    print("  Small MLP: 2-5×")
    print("  CNN models: 3-8×")
    print("  Transformer (small): 2-4×")
    print("  Key benefit: no Python overhead, compiled inference graph")
    print()
    print("PRODUCTION DEPLOYMENT STACK:")
    print("  Training: PyTorch (flexible, GPU)")
    print("  Export: ONNX (portable format)")
    print("  Serving: ONNX Runtime or TensorRT (fast CPU/GPU inference)")
    print("  Container: Docker + FastAPI (scalable)")
    print("  Orchestration: Kubernetes (multi-instance, auto-scaling)")

print()
print("FULL PRODUCTION PIPELINE:")
print("  1. Train model with MLflow tracking (experiment versioning)")
print("  2. Evaluate on test set, log metrics to MLflow")
print("  3. If test metrics pass threshold → export to ONNX")
print("  4. Register ONNX model in MLflow Model Registry with version tag")
print("  5. Deploy to FastAPI server via model registry activation")
print("  6. Monitor with KS drift detection (module 16/01)")
print("  7. Trigger retraining when drift detected")